# Notebook 03 - Bronze Layer

## Objective

Load all raw Healthcare & Pharma CSV files from the Databricks Volume into Bronze Delta Tables.

### Source
Databricks Volume

### Target
Delta Tables (Bronze Layer)

### Files

- biotech_funding.csv
- clinical_trials.csv
- disease_burden.csv
- drug_approvals.csv
- pharma_companies_financials.csv

### Metadata Added

- source_file
- load_timestamp

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
# Volume Path
volume_path = "/Volumes/dbacademy/default/raw_data"

# Catalog
catalog = "workspace"

# Bronze Schema
schema = "default"

In [0]:
files = [
    {
        "file_name": "biotech_funding.csv",
        "table_name": "biotech_bronze"
    },
    {
        "file_name": "clinical_trials.csv",
        "table_name": "clinical_trials_bronze"
    },
    {
        "file_name": "disease_burden.csv",
        "table_name": "disease_burden_bronze"
    },
    {
        "file_name": "drug_approvals.csv",
        "table_name": "drug_approvals_bronze"
    },
    {
        "file_name": "pharma_companies_financials.csv",
        "table_name": "pharma_companies_bronze"
    }
]

In [0]:
for file in files:

    print("=" * 70)
    print(f"Processing : {file['file_name']}")

    # Read CSV
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{volume_path}/{file['file_name']}")
    )

    # Add Metadata
    bronze_df = (
        df
        .withColumn("source_file", lit(file["file_name"]))
        .withColumn("load_timestamp", current_timestamp())
    )

    # Write Bronze Table
    bronze_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(
            f"{catalog}.{schema}.{file['table_name']}"
        )

    print(f"Table Created : {file['table_name']}")

    print(f"Rows Loaded : {bronze_df.count()}")

print("=" * 70)
print("Bronze Layer Completed Successfully")

In [0]:
%sql

SHOW TABLES IN workspace.default;